# Install Streamlit and UI packages

In [1]:
# Install packages required for the PhishDetectAI Streamlit UI

!pip install -q streamlit plotly seaborn matplotlib chromadb sentence-transformers

# Mount Google drive

In [2]:
# Mount Google Drive to access the PhishDetectAI project files

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Check the PhishDetectAI project folder

In [3]:
# Check whether the PhishDetectAI project folder exists in Google Drive

import os

BASE_DIR = "/content/drive/MyDrive/PhishDetectAI"

print("Project folder exists:", os.path.exists(BASE_DIR))

if os.path.exists(BASE_DIR):
    print("Project contents:")
    print(os.listdir(BASE_DIR))

Project folder exists: True
Project contents:
['datasets', 'models', 'notebooks', 'embeddings', 'results', 'training_results_correct', 'rag_chroma_correct', 'app.py']


# Check ChromaDB and Sentence Transformers

In [4]:
# Check that ChromaDB and Sentence Transformers are installed correctly

import chromadb
import sentence_transformers

print("ChromaDB:", chromadb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

ChromaDB: 1.5.9
Sentence Transformers: 5.7.0


# Check the RAG database

In [5]:
# Check that the saved ChromaDB RAG database and phishing_emails collection are available

import chromadb
import os

CHROMA_PATH = os.path.join(
    BASE_DIR,
    "rag_chroma_correct"
)

print("ChromaDB folder exists:", os.path.exists(CHROMA_PATH))

# Connect to the persistent ChromaDB database
client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

# Check available collections
collections = client.list_collections()

print("Available collections:")

for collection in collections:
    print("-", collection.name)

ChromaDB folder exists: True
Available collections:
- phishing_emails


# Check how many emails are stored

In [6]:
# Check how many phishing/legitimate emails are stored in the RAG collection

collection = client.get_collection(
    name="phishing_emails"
)

print("Collection name:", collection.name)
print("Number of stored emails:", collection.count())

Collection name: phishing_emails
Number of stored emails: 23813


# Load the fine-tuned Qwen model

In [7]:
!pip install unsloth

In [8]:
# ============================================================
# Load the fine-tuned Qwen model for both prediction pipelines
# ============================================================

import unsloth
from unsloth import FastLanguageModel

MODEL_PATH = "/content/drive/MyDrive/PhishDetectAI/models/qwen_phishing_unsloth_correct"

print("Loading fine-tuned Qwen model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=1024,
    load_in_4bit=True
)

FastLanguageModel.for_inference(model)

print("Fine-tuned Qwen loaded successfully!")

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1607: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading fine-tuned Qwen model...
==((====))==  Unsloth 2026.9.12: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.9.12 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Fine-tuned Qwen loaded successfully!


# Load the embedding model

In [9]:
# Load the Sentence Transformer model used to create the RAG embeddings

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


# Test RAG retrieval

In [10]:
# Test RAG retrieval using a sample phishing email

sample_email = """
Subject: Urgent Account Verification Required

Dear Customer,

Your account requires immediate verification.
Please click the link below to confirm your account information.
Failure to verify your account within 24 hours may result in suspension.

Thank you.
"""

# Convert the email into an embedding
query_embedding = embedding_model.encode(
    sample_email,
    show_progress_bar=False
).tolist()

# Search for the 3 most similar emails in ChromaDB
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

# Display retrieved results
print("=" * 70)
print("RAG RETRIEVAL TEST")
print("=" * 70)

for i in range(3):

    label = results["metadatas"][0][i]["label"]
    distance = results["distances"][0][i]
    document = results["documents"][0][i]

    print(f"\nEvidence {i + 1}")
    print("-" * 70)
    print("Label:", label)
    print("Distance:", round(distance, 4))
    print("Email preview:")
    print(document[:500])

RAG RETRIEVAL TEST

Evidence 1
----------------------------------------------------------------------
Label: phishing
Distance: 0.664
Email preview:
Subject: urgent safeharbor department warning

Body: urgent
safeharbor department warning
we recently have determined that different computers have logged into your
ebay account , and multiple password failures were present before the login one
of our customer service employees has already tryed to telephonically reach you .
as our employee did not manage to reach you , this email has been sent to your
notice . therefore your account has been temporary suspended . we need you to
confirm y

Evidence 2
----------------------------------------------------------------------
Label: phishing
Distance: 0.7183
Email preview:
Subject: note ! citibank account suspend in process

Body: dear customer :
recently there have been a large number of cyber attacks pointing our database servers . in order to safeguard your account , we require you to sign on

# Create Two prediction functions

In [11]:
# ============================================================
# Create prediction functions for:
# 1. Fine-tuned Qwen
# 2. RAG + Fine-tuned Qwen
# ============================================================

import torch


# ============================================================
# System prompt used during fine-tuning
# ============================================================

SYSTEM_PROMPT = (
    "You are a cybersecurity expert specializing in phishing detection. "
    "Analyze the provided email, message, or URL. "
    "Determine whether it is a phishing attempt or a legitimate communication. "
    "Respond with only one word: 'phishing' or 'legitimate'."
)


# ============================================================
# Helper function
# Converts generated text into a valid classification label
# ============================================================

def extract_label(generated_text):

    generated_text = generated_text.strip().lower()

    if "phishing" in generated_text:
        return "phishing"

    elif "legitimate" in generated_text:
        return "legitimate"

    else:
        return "invalid"


# ============================================================
# 1. Direct Fine-Tuned Qwen
# ============================================================

def predict_with_qwen(email_text):

    # Create the same message structure used during fine-tuning
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": email_text
        }
    ]

    # Use Qwen's chat template
    formatted_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize the prompt
    inputs = tokenizer(
        formatted_input,
        return_tensors="pt"
    ).to(model.device)

    # Generate the classification
    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only the newly generated tokens
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # Convert generated text to label
    prediction = extract_label(generated_text)

    return prediction, generated_text


# ============================================================
# 2. RAG + Fine-Tuned Qwen
# ============================================================

# ============================================================
# RAG + Fine-Tuned Qwen
# Version with shorter evidence to stay within 1024 tokens
# ============================================================

def predict_with_rag_qwen(email_text, n_results=3):

    # --------------------------------------------------------
    # Step 1: Create embedding for the input email
    # --------------------------------------------------------

    query_embedding = embedding_model.encode(
        email_text,
        show_progress_bar=False
    ).tolist()


    # --------------------------------------------------------
    # Step 2: Retrieve similar emails from ChromaDB
    # --------------------------------------------------------

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )


    # --------------------------------------------------------
    # Step 3: Build SHORT retrieved evidence
    #
    # We only use the first 300 characters from each
    # retrieved email so the complete prompt stays small.
    # --------------------------------------------------------

    evidence_text = ""

    for i in range(n_results):

        retrieved_email = results["documents"][0][i]
        retrieved_label = results["metadatas"][0][i]["label"]

        # Keep only a short excerpt
        email_excerpt = retrieved_email[:300]

        evidence_text += (
            f"\n--- Evidence {i + 1} ---\n"
            f"Label: {retrieved_label}\n"
            f"Email excerpt: {email_excerpt}\n"
        )


    # --------------------------------------------------------
    # Step 4: Create the RAG prompt
    # --------------------------------------------------------

    rag_user_content = f"""
Email to classify:

{email_text[:2500]}

Retrieved similar email evidence:

{evidence_text}

Classify the original email.
Respond with only one word: 'phishing' or 'legitimate'.
"""


    # --------------------------------------------------------
    # Step 5: Create Qwen messages
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": rag_user_content
        }
    ]


    # --------------------------------------------------------
    # Step 6: Apply the SAME Qwen chat template
    # used during fine-tuning
    # --------------------------------------------------------

    formatted_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # --------------------------------------------------------
    # Step 7: Tokenize
    # --------------------------------------------------------

    inputs = tokenizer(
        formatted_input,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)


    # --------------------------------------------------------
    # Step 8: Generate prediction
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )


    # --------------------------------------------------------
    # Step 9: Extract only generated tokens
    # --------------------------------------------------------

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


    # --------------------------------------------------------
    # Step 10: Convert generated text into a label
    # --------------------------------------------------------

    prediction = extract_label(generated_text)


    return prediction, generated_text, results

# Test both with same email

In [12]:
# ============================================================
# Test both prediction pipelines
# ============================================================

test_email = """
Subject: Urgent Account Verification Required

Dear Customer,

Your account requires immediate verification.
Please click the link below to confirm your account information.
Failure to verify your account within 24 hours may result in suspension.

Thank you.
"""


# ============================================================
# Direct Fine-Tuned Qwen
# ============================================================

qwen_prediction, qwen_generated = predict_with_qwen(
    test_email
)

print("=" * 70)
print("DIRECT FINE-TUNED QWEN")
print("=" * 70)

print("Prediction:", qwen_prediction)
print("Generated:", qwen_generated)


# ============================================================
# RAG + Fine-Tuned Qwen
# ============================================================

rag_prediction, rag_generated, rag_results = predict_with_rag_qwen(
    test_email,
    n_results=3
)

print("\n" + "=" * 70)
print("RAG + FINE-TUNED QWEN")
print("=" * 70)

print("Prediction:", rag_prediction)
print("Generated:", rag_generated)


# ============================================================
# Show retrieved evidence
# ============================================================

print("\n" + "=" * 70)
print("RETRIEVED EVIDENCE")
print("=" * 70)

for i in range(3):

    label = rag_results["metadatas"][0][i]["label"]
    distance = rag_results["distances"][0][i]

    print(f"\nEvidence {i + 1}")
    print("Label:", label)
    print("Distance:", round(distance, 4))

Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DIRECT FINE-TUNED QWEN
Prediction: phishing
Generated: phishing

RAG + FINE-TUNED QWEN
Prediction: phishing
Generated: phishing

RETRIEVED EVIDENCE

Evidence 1
Label: phishing
Distance: 0.664

Evidence 2
Label: phishing
Distance: 0.7183

Evidence 3
Label: phishing
Distance: 0.7524


# Final inference

In [13]:
# ============================================================
# FINAL INFERENCE FUNCTIONS
# Fine-tuned Qwen + RAG + Fine-tuned Qwen
# ============================================================

import torch


# ============================================================
# System prompt
# This is the SAME prompt used during Qwen fine-tuning.
# ============================================================

SYSTEM_PROMPT = (
    "You are a cybersecurity expert specializing in phishing detection. "
    "Analyze the provided email, message, or URL. "
    "Determine whether it is a phishing attempt or a legitimate communication. "
    "Respond with only one word: 'phishing' or 'legitimate'."
)


# ============================================================
# Helper function
# Convert Qwen's generated text into a valid label.
# ============================================================

def extract_label(generated_text):

    generated_text = generated_text.strip().lower()

    if "phishing" in generated_text:
        return "phishing"

    elif "legitimate" in generated_text:
        return "legitimate"

    else:
        return "invalid"


# ============================================================
# PIPELINE 1
# Fine-Tuned Qwen only
# ============================================================

def predict_with_qwen(email_text):

    # --------------------------------------------------------
    # Create the same message structure used during training
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": email_text
        }
    ]


    # --------------------------------------------------------
    # Apply Qwen's chat template
    # --------------------------------------------------------

    formatted_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # --------------------------------------------------------
    # Tokenize the input
    # --------------------------------------------------------

    inputs = tokenizer(
        formatted_input,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)


    # --------------------------------------------------------
    # Generate the answer
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )


    # --------------------------------------------------------
    # Extract only the newly generated tokens
    # --------------------------------------------------------

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


    # --------------------------------------------------------
    # Convert generated text into a classification label
    # --------------------------------------------------------

    prediction = extract_label(generated_text)


    return prediction, generated_text


# ============================================================
# PIPELINE 2
# RAG + Fine-Tuned Qwen
# ============================================================

def predict_with_rag_qwen(email_text, n_results=3):

    # --------------------------------------------------------
    # STEP 1: Create an embedding for the input email
    # --------------------------------------------------------

    query_embedding = embedding_model.encode(
        email_text,
        show_progress_bar=False
    ).tolist()


    # --------------------------------------------------------
    # STEP 2: Retrieve similar emails from ChromaDB
    # --------------------------------------------------------

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )


    # --------------------------------------------------------
    # STEP 3: Create compact evidence
    #
    # We only use 300 characters from each retrieved email.
    # This keeps the Qwen prompt within the 1024-token limit.
    # --------------------------------------------------------

    evidence_text = ""

    for i in range(n_results):

        retrieved_email = results["documents"][0][i]

        retrieved_label = (
            results["metadatas"][0][i]["label"]
        )

        email_excerpt = retrieved_email[:300]

        evidence_text += (
            f"\n--- Evidence {i + 1} ---\n"
            f"Label: {retrieved_label}\n"
            f"Email excerpt: {email_excerpt}\n"
        )


    # --------------------------------------------------------
    # STEP 4: Create the RAG user prompt
    # --------------------------------------------------------

    rag_user_content = f"""
Email to classify:

{email_text[:2500]}

Retrieved similar email evidence:

{evidence_text}

Classify the original email.
Respond with only one word: 'phishing' or 'legitimate'.
"""


    # --------------------------------------------------------
    # STEP 5: Create Qwen messages
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": rag_user_content
        }
    ]


    # --------------------------------------------------------
    # STEP 6: Apply the Qwen chat template
    # --------------------------------------------------------

    formatted_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # --------------------------------------------------------
    # STEP 7: Tokenize
    # --------------------------------------------------------

    inputs = tokenizer(
        formatted_input,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)


    # --------------------------------------------------------
    # STEP 8: Generate final classification
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )


    # --------------------------------------------------------
    # STEP 9: Extract generated answer
    # --------------------------------------------------------

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


    # --------------------------------------------------------
    # STEP 10: Convert to final label
    # --------------------------------------------------------

    prediction = extract_label(generated_text)


    # Return both prediction and retrieved evidence
    return prediction, generated_text, results


print("=" * 70)
print("FINAL INFERENCE FUNCTIONS CREATED")
print("=" * 70)

print("✓ Direct Fine-Tuned Qwen")
print("✓ RAG + Fine-Tuned Qwen")
print("✓ ChromaDB retrieval")
print("✓ 1024-token input protection")

FINAL INFERENCE FUNCTIONS CREATED
✓ Direct Fine-Tuned Qwen
✓ RAG + Fine-Tuned Qwen
✓ ChromaDB retrieval
✓ 1024-token input protection


# Final test

In [14]:
# ============================================================
# FINAL TEST
# Test both prediction pipelines
# ============================================================

test_email = """
Subject: Urgent Account Verification Required

Dear Customer,

Your account requires immediate verification.
Please click the link below to confirm your account information.
Failure to verify your account within 24 hours may result in suspension.

Thank you.
"""


# ============================================================
# PIPELINE 1: Fine-Tuned Qwen
# ============================================================

qwen_prediction, qwen_generated = predict_with_qwen(
    test_email
)


# ============================================================
# PIPELINE 2: RAG + Fine-Tuned Qwen
# ============================================================

rag_prediction, rag_generated, rag_results = predict_with_rag_qwen(
    test_email,
    n_results=3
)


# ============================================================
# Display Direct Qwen Result
# ============================================================

print("=" * 70)
print("PIPELINE 1 — FINE-TUNED QWEN")
print("=" * 70)

print("Prediction:", qwen_prediction)
print("Generated text:", qwen_generated)


# ============================================================
# Display RAG + Qwen Result
# ============================================================

print("\n" + "=" * 70)
print("PIPELINE 2 — RAG + FINE-TUNED QWEN")
print("=" * 70)

print("Prediction:", rag_prediction)
print("Generated text:", rag_generated)


# ============================================================
# Display Retrieved Evidence
# ============================================================

print("\n" + "=" * 70)
print("RAG RETRIEVED EVIDENCE")
print("=" * 70)

for i in range(3):

    label = rag_results["metadatas"][0][i]["label"]
    distance = rag_results["distances"][0][i]

    print(f"\nEvidence {i + 1}")
    print("-" * 50)
    print("Label:", label)
    print("Distance:", round(distance, 4))

Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PIPELINE 1 — FINE-TUNED QWEN
Prediction: phishing
Generated text: phishing

PIPELINE 2 — RAG + FINE-TUNED QWEN
Prediction: phishing
Generated text: phishing

RAG RETRIEVED EVIDENCE

Evidence 1
--------------------------------------------------
Label: phishing
Distance: 0.664

Evidence 2
--------------------------------------------------
Label: phishing
Distance: 0.7183

Evidence 3
--------------------------------------------------
Label: phishing
Distance: 0.7524


# Create Streamlit App

In [15]:
# ============================================================
# CREATE FINAL STREAMLIT APP
# ============================================================

import os

APP_PATH = "/content/drive/MyDrive/PhishDetectAI/app.py"

with open(APP_PATH, "w", encoding="utf-8") as f:

    f.write(r'''
# ============================================================
# PhishDetectAI
# Streamlit application
#
# Pages:
# 1. Email Detection
# 2. Model Evaluation
#
# Models:
# - Fine-Tuned Qwen
# - RAG + Fine-Tuned Qwen
# - TF-IDF + Logistic Regression
# ============================================================

import os
import torch
import streamlit as st
import chromadb

import unsloth
from unsloth import FastLanguageModel

from sentence_transformers import SentenceTransformer


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="PhishDetectAI",
    page_icon="🛡️",
    layout="wide"
)


# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/PhishDetectAI"

MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "qwen_phishing_unsloth_correct"
)

CHROMA_PATH = os.path.join(
    BASE_DIR,
    "rag_chroma_correct"
)


# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = (
    "You are a cybersecurity expert specializing in phishing detection. "
    "Analyze the provided email, message, or URL. "
    "Determine whether it is a phishing attempt or a legitimate communication. "
    "Respond with only one word: 'phishing' or 'legitimate'."
)


# ============================================================
# LOAD QWEN MODEL
# ============================================================

@st.cache_resource
def load_qwen_model():

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=1024,
        load_in_4bit=True
    )

    FastLanguageModel.for_inference(model)

    return model, tokenizer


# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

@st.cache_resource
def load_embedding_model():

    return SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )


# ============================================================
# LOAD CHROMADB COLLECTION
# ============================================================

@st.cache_resource
def load_collection():

    client = chromadb.PersistentClient(
        path=CHROMA_PATH
    )

    collection = client.get_collection(
        name="phishing_emails"
    )

    return collection


# ============================================================
# LOAD MODELS
# ============================================================

with st.spinner("Loading AI models..."):

    model, tokenizer = load_qwen_model()

    embedding_model = load_embedding_model()

    collection = load_collection()


# ============================================================
# HELPER — EXTRACT LABEL
# ============================================================

def extract_label(generated_text):

    generated_text = generated_text.strip().lower()

    if "phishing" in generated_text:
        return "phishing"

    elif "legitimate" in generated_text:
        return "legitimate"

    else:
        return "invalid"


# ============================================================
# DIRECT FINE-TUNED QWEN
# ============================================================

def predict_with_qwen(email_text):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": email_text
        }
    ]

    formatted_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_input,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = (
        outputs[0][inputs.input_ids.shape[1]:]
    )

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    prediction = extract_label(
        generated_text
    )

    return prediction


# ============================================================
# RAG + FINE-TUNED QWEN
# ============================================================

def predict_with_rag_qwen(
    email_text,
    n_results=3
):

    # --------------------------------------------------------
    # STEP 1 — CREATE EMAIL EMBEDDING
    # --------------------------------------------------------

    query_embedding = embedding_model.encode(
        email_text,
        show_progress_bar=False
    ).tolist()

    # --------------------------------------------------------
    # STEP 2 — RETRIEVE TOP SIMILAR EMAILS
    # --------------------------------------------------------

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    # --------------------------------------------------------
    # STEP 3 — CREATE EVIDENCE
    # --------------------------------------------------------

    evidence_text = ""

    retrieved_count = len(
        results["documents"][0]
    )

    for i in range(retrieved_count):

        retrieved_email = (
            results["documents"][0][i]
        )

        retrieved_label = (
            results["metadatas"][0][i]["label"]
        )

        email_excerpt = retrieved_email[:300]

        evidence_text += (
            f"\n--- Evidence {i + 1} ---\n"
            f"Label: {retrieved_label}\n"
            f"Email excerpt: {email_excerpt}\n"
        )

    # --------------------------------------------------------
    # STEP 4 — CREATE RAG PROMPT
    # --------------------------------------------------------

    rag_user_content = f"""
Email to classify:

{email_text[:2500]}

Retrieved similar email evidence:

{evidence_text}

Classify the original email.

Respond with only one word: 'phishing' or 'legitimate'.
"""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": rag_user_content
        }
    ]

    # --------------------------------------------------------
    # STEP 5 — APPLY CHAT TEMPLATE
    # --------------------------------------------------------

    formatted_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # --------------------------------------------------------
    # STEP 6 — TOKENIZE
    # --------------------------------------------------------

    inputs = tokenizer(
        formatted_input,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    # --------------------------------------------------------
    # STEP 7 — GENERATE
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    # --------------------------------------------------------
    # STEP 8 — DECODE
    # --------------------------------------------------------

    generated_ids = (
        outputs[0][inputs.input_ids.shape[1]:]
    )

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # STEP 9 — EXTRACT LABEL
    # --------------------------------------------------------

    prediction = extract_label(
        generated_text
    )

    return prediction, results


# ============================================================
# SIDEBAR NAVIGATION
# ============================================================

st.sidebar.title("🛡️ PhishDetectAI")

page = st.sidebar.radio(
    "Navigation",
    [
        "🔍 Email Detection",
        "📊 Model Evaluation"
    ]
)


# ============================================================
# PAGE 1 — EMAIL DETECTION
# ============================================================

if page == "🔍 Email Detection":

    st.title("🛡️ PhishDetectAI")

    st.write(
        "AI-powered phishing email detection using "
        "Fine-Tuned Qwen and Retrieval-Augmented Generation."
    )

    st.divider()

    st.subheader("📧 Analyze an Email")

    # --------------------------------------------------------
    # SESSION STATE
    # --------------------------------------------------------

    if "email_widget_key" not in st.session_state:
        st.session_state.email_widget_key = 0

    if "email_input" not in st.session_state:
        st.session_state.email_input = ""

    if "analysis_results_displayed" not in st.session_state:
        st.session_state.analysis_results_displayed = False

    if "qwen_prediction" not in st.session_state:
        st.session_state.qwen_prediction = None

    if "rag_prediction" not in st.session_state:
        st.session_state.rag_prediction = None

    if "rag_results" not in st.session_state:
        st.session_state.rag_results = None

    # --------------------------------------------------------
    # DYNAMIC TEXT AREA KEY
    #
    # Changing the key forces Streamlit to create a new
    # empty text area after Clear Email.
    # --------------------------------------------------------

    email_widget_key = (
        f"email_text_{st.session_state.email_widget_key}"
    )

    email_text = st.text_area(
        "Enter the email content:",
        height=250,
        placeholder=(
            "Subject: Your account requires verification\n\n"
            "Body: Please click the link below..."
        ),
        key=email_widget_key
    )

    # --------------------------------------------------------
    # BUTTONS
    # --------------------------------------------------------

    col_analyze, col_clear = st.columns([0.7, 0.3])

    with col_analyze:

        analyze_button = st.button(
            "🔍 Analyze Email",
            type="primary",
            use_container_width=True
        )

    with col_clear:

        clear_button = st.button(
            "Clear Email",
            use_container_width=True
        )

    # --------------------------------------------------------
    # ANALYZE
    # --------------------------------------------------------

    if analyze_button:

        if not email_text.strip():

            st.warning(
                "Please enter an email before analyzing."
            )

            st.session_state.analysis_results_displayed = False

        else:

            st.session_state.email_input = email_text

            with st.spinner(
                "Analyzing email with both pipelines..."
            ):

                # --------------------------------------------
                # FINE-TUNED QWEN
                # --------------------------------------------

                qwen_prediction = predict_with_qwen(
                    email_text
                )

                # --------------------------------------------
                # RAG + QWEN
                # --------------------------------------------

                rag_prediction, rag_results = (
                    predict_with_rag_qwen(
                        email_text,
                        n_results=3
                    )
                )

            # ----------------------------------------------
            # SAVE RESULTS
            # ----------------------------------------------

            st.session_state.qwen_prediction = (
                qwen_prediction
            )

            st.session_state.rag_prediction = (
                rag_prediction
            )

            st.session_state.rag_results = (
                rag_results
            )

            st.session_state.analysis_results_displayed = True

    # --------------------------------------------------------
    # CLEAR EMAIL
    # --------------------------------------------------------

    if clear_button:

        # Clear stored email
        st.session_state.email_input = ""

        # Clear predictions
        st.session_state.qwen_prediction = None

        st.session_state.rag_prediction = None

        st.session_state.rag_results = None

        # Hide results
        st.session_state.analysis_results_displayed = False

        # Change widget key
        st.session_state.email_widget_key += 1

        # Rerun
        st.rerun()

    # --------------------------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------------------------

    if st.session_state.analysis_results_displayed:

        st.divider()

        st.subheader("🤖 Model Results")

        col1_res, col2_res = st.columns(2)

        # ====================================================
        # FINE-TUNED QWEN
        # ====================================================

        with col1_res:

            st.markdown(
                "### 🔵 Fine-Tuned Qwen"
            )

            if (
                st.session_state.qwen_prediction
                == "phishing"
            ):

                st.error(
                    "🚨 PHISHING"
                )

            elif (
                st.session_state.qwen_prediction
                == "legitimate"
            ):

                st.success(
                    "✅ LEGITIMATE"
                )

            else:

                st.warning(
                    "⚠️ INVALID RESPONSE"
                )

        # ====================================================
        # RAG + QWEN
        # ====================================================

        with col2_res:

            st.markdown(
                "### 🟣 RAG + Fine-Tuned Qwen"
            )

            if (
                st.session_state.rag_prediction
                == "phishing"
            ):

                st.error(
                    "🚨 PHISHING"
                )

            elif (
                st.session_state.rag_prediction
                == "legitimate"
            ):

                st.success(
                    "✅ LEGITIMATE"
                )

            else:

                st.warning(
                    "⚠️ INVALID RESPONSE"
                )

        # ====================================================
        # RAG EVIDENCE
        # ====================================================

        st.divider()

        st.subheader(
            "🔎 RAG Retrieved Evidence"
        )

        st.caption(
            "These emails were retrieved from the ChromaDB "
            "knowledge base and provided to Qwen as "
            "supporting evidence."
        )

        rag_results = st.session_state.rag_results

        if rag_results is not None:

            retrieved_count = len(
                rag_results["documents"][0]
            )

            for i in range(retrieved_count):

                label = (
                    rag_results["metadatas"][0][i]["label"]
                )

                distance = (
                    rag_results["distances"][0][i]
                )

                document = (
                    rag_results["documents"][0][i]
                )

                with st.expander(
                    f"Evidence {i + 1} — {label}"
                ):

                    st.write(
                        f"**Retrieved label:** {label}"
                    )

                    st.write(
                        f"**Distance:** {distance:.4f}"
                    )

                    st.text_area(
                        f"Retrieved email {i + 1}",
                        document,
                        height=180,
                        key=f"evidence_{i}"
                    )


# ============================================================
# PAGE 2 — MODEL EVALUATION
# ============================================================

elif page == "📊 Model Evaluation":

    st.title("📊 Model Evaluation")

    st.write(
        "Performance comparison of the three phishing "
        "detection approaches."
    )

    st.divider()

    st.subheader(
        "📈 Model Comparison on 2,977 Emails"
    )

    st.write(
        "All three models were evaluated on the same "
        "2,977-email test set."
    )

    # --------------------------------------------------------
    # SAVED EVALUATION RESULTS
    # --------------------------------------------------------

    comparison_data = {
        "Model": [
            "TF-IDF + Logistic Regression",
            "Fine-Tuned Qwen",
            "RAG + Qwen"
        ],

        "Accuracy": [
            "98.62%",
            "99.20%",
            "98.02%"
        ],

        "Precision": [
            "97.50%",
            "99.65%",
            "97.15%"
        ],

        "Recall": [
            "99.65%",
            "98.82%",
            "98.70%"
        ],

        "F1-Score": [
            "98.56%",
            "99.23%",
            "97.92%"
        ],

        "Invalid Predictions": [
            0,
            242,
            55
        ]
    }

    st.dataframe(
        comparison_data,
        use_container_width=True,
        hide_index=True
    )

    # --------------------------------------------------------
    # METRIC EXPLANATION
    # --------------------------------------------------------

    st.divider()

    st.subheader(
        "📚 Evaluation Metrics"
    )

    col1, col2 = st.columns(2)

    with col1:

        st.markdown(
            """
            **Accuracy**

            Percentage of predictions that were correct.

            **Precision**

            Of the emails predicted as phishing,
            how many were actually phishing.

            **Recall**

            Of the actual phishing emails,
            how many were detected.
            """
        )

    with col2:

        st.markdown(
            """
            **F1-Score**

            A combined measure of precision and recall.

            **Invalid Predictions**

            Model responses that did not produce
            a valid `phishing` or `legitimate` label.
            """
        )

    # --------------------------------------------------------
    # EVALUATION SETUP
    # --------------------------------------------------------

    st.divider()

    st.subheader(
        "🧪 Evaluation Setup"
    )

    st.write(
        """
        **Test set:** 2,977 emails

        **Models compared:**
        - TF-IDF + Logistic Regression
        - Fine-Tuned Qwen
        - RAG + Fine-Tuned Qwen

        The same test split was used for the comparison
        to make the model results directly comparable.
        """
    )

    st.caption(
        "Results shown here were calculated previously in "
        "evaluation.ipynb. Opening this page does not "
        "rerun model evaluation."
    )
''')

print("=" * 70)
print("FINAL TWO-PAGE STREAMLIT APP CREATED")
print("=" * 70)
print("App path:")
print(APP_PATH)
print("=" * 70)

FINAL TWO-PAGE STREAMLIT APP CREATED
App path:
/content/drive/MyDrive/PhishDetectAI/app.py


# Start Streamlit server

In [16]:
# ============================================================
# START STREAMLIT
# ============================================================

import subprocess
import time

# ------------------------------------------------------------
# Stop previous Streamlit processes
# ------------------------------------------------------------

subprocess.run(
    ["pkill", "-f", "streamlit"],
    stderr=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL
)

time.sleep(2)

# ------------------------------------------------------------
# Start Streamlit
# ------------------------------------------------------------

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# ------------------------------------------------------------
# Wait for server
# ------------------------------------------------------------

time.sleep(5)

print("=" * 70)
print("STREAMLIT SERVER STARTED")
print("=" * 70)
print(f"PID: {process.pid}")
print("Port: 8501")
print("App path:", APP_PATH)
print("=" * 70)

STREAMLIT SERVER STARTED
PID: 4320
Port: 8501
App path: /content/drive/MyDrive/PhishDetectAI/app.py


# Open Streamlit UI

In [17]:
# ============================================================
# OPEN STREAMLIT IN COLAB
# ============================================================

from google.colab import output
from google.colab.output import eval_js


# Create a Colab proxy URL for Streamlit
proxy_url = eval_js(
    "google.colab.kernel.proxyPort(8501)"
)


print("Streamlit URL:")
print(proxy_url)


# Display the application inside Colab
output.serve_kernel_port_as_iframe(
    8501,
    height=850
)

Streamlit URL:
https://8501-gpu-t4-s-kkb-usw1b0-25xv93llsomhi-b.us-west1-0.prod.colab.dev


<IPython.core.display.Javascript object>